In [1]:
!pip install -U transformers faiss-gpu-cu11 ultralytics accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.2 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: a

# Pre-Processing

## Create the Dataset Class

In [2]:
import os
from PIL import Image
from torch.utils.data import Dataset

class DeepFashionDataset(Dataset):
    def __init__(self, partition_file, img_base_dir, split, transform=None, bbox_dict=None):
        """
        Args:
            partition_file (str): Path to list_eval_partition.txt
            img_base_dir (str): Path to the folder containing the 'img' directory
            split (str): One of 'train', 'gallery', or 'query'
            transform (callable, optional): PyTorch transforms to apply to the image
            bbox_dict (dict, optional): Dictionary containing bounding boxes
        """
        self.img_base_dir = img_base_dir
        self.transform = transform
        self.bbox_dict = bbox_dict
        
        # Load the partition file
        df = pd.read_csv(partition_file, sep=r'\s+', skiprows=1)
        df.columns = df.columns.str.strip()
        
        # Filter for the requested split
        self.data = df[df['evaluation_status'] == split].reset_index(drop=True)
        print(f"Initialized {split} dataset with {len(self.data)} images.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        rel_path = row['image_name']
        item_id = row['item_id']
        
        # Construct full image path and load it
        img_path = os.path.join(self.img_base_dir, rel_path)
        
        # Convert to RGB to ensure consistency (some images might be grayscale/RGBA)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        # Get bounding box if available
        bbox = []
        if self.bbox_dict and rel_path in self.bbox_dict:
            bbox = self.bbox_dict[rel_path]
            
        return {
            'image': image,
            'item_id': item_id,
            'rel_path': rel_path,
            'bbox': bbox
        }

## Cropping Function for YOLO

In [3]:
import os
import random
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from torch.optim import AdamW
from ultralytics import YOLO
from tqdm.notebook import tqdm
from IPython.display import display

# --- KAGGLE DEVICE CONFIG ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- PATHS ---
PARTITION_FILE = "/kaggle/input/datasets/dveers/vr-final-ds/list_eval_partition.txt"
BBOX_FILE = "/kaggle/input/datasets/dveers/vr-final-ds/list_bbox_inshop.txt"
BASE_DIR = os.path.abspath("/kaggle/input/datasets/dveers/vr-final-ds")
YOLO_MODEL_PATH = '/kaggle/input/models/dveers/yolov11-vr-final/pytorch/default/3/runs/detect/train/weights/best.pt'

# --- LOAD YOLO ---
print("Loading fine-tuned YOLO...")
yolo_model = YOLO(YOLO_MODEL_PATH)

def get_cropped_image(img_path, bbox=None):
    """Crops the image using bounding boxes or YOLO."""
    img = Image.open(img_path).convert('RGB')
    
    if bbox is not None and len(bbox) == 4:
        return img.crop((bbox[0], bbox[1], bbox[2], bbox[3]))
        
    # FIX: Added device targeting and half-precision for Kaggle GPU speed
    use_half = True if DEVICE == "cuda" else False
    results = yolo_model(img, conf=0.25, verbose=False, half=use_half)
    
    if len(results[0].boxes) > 0:
        boxes = results[0].boxes.data
        best_box = boxes[boxes[:, 4].argmax()] 
        x1, y1, x2, y2 = best_box[:4].tolist()
        return img.crop((x1, y1, x2, y2))
        
    return img

# --- Quick Test ---
# test_rel_path = 'img/WOMEN/Leggings/id_00000225/03_2_side.jpg'
# test_abs_path = os.path.join(BASE_DIR, test_rel_path)
# if os.path.exists(test_abs_path):
#     cropped = get_cropped_image(test_abs_path)
#     display(cropped) 
# else:
#     print(f"Test image not found at: {test_abs_path}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading fine-tuned YOLO...


# Index Builders

FAISS Indices for Ablation Part C are already prepped in VR_final_AB.ipynb

# Ablation Study

In [4]:
import pandas as pd
import numpy as np
import torch
import os
import faiss
from tqdm.notebook import tqdm
from IPython.display import display, HTML

# 1. Point to your partition file

BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
PARTITION_FILE = os.path.join(BASE_DIR, "list_eval_partition.txt")

print("Loading query partition...")

# 2. Load the text file into a pandas DataFrame
df_part = pd.read_csv(PARTITION_FILE, sep=r'\s+', skiprows=1)

# 3. Clean the column names (scrubs hidden Windows characters)
df_part.columns = df_part.columns.str.strip()

# 4. Filter to create query_df (grabbing ONLY the 'query' status images)
query_df = df_part[df_part['evaluation_status'] == 'query'].copy()

# 5. Clean the image paths just to be safe
query_df['image_name'] = query_df['image_name'].str.strip()

print(f"Success! Loaded {len(query_df)} Query images ready for testing.")

def calculate_metrics(retrieved_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    results = {'Recall': {}, 'NDCG': {}, 'mAP': {}}
    for k in k_values:
        top_k = retrieved_item_ids[:k]
        relevance = [1 if item == ground_truth_id else 0 for item in top_k]
        
        results['Recall'][k] = 1 if sum(relevance) > 0 else 0
        dcg = sum([rel / np.log2(idx + 2) for idx, rel in enumerate(relevance)])
        results['NDCG'][k] = dcg / 1.0  
        precisions = [hits / (i + 1) for i, (rel, hits) in enumerate(zip(relevance, np.cumsum(relevance))) if rel == 1]
        results['mAP'][k] = sum(precisions) / min(len(top_k), sum(relevance) + 1e-6) if precisions else 0
    return results

Loading query partition...
Success! Loaded 14218 Query images ready for testing.


In [5]:
import os
import json
import torch
import faiss
import gc
import pandas as pd
import numpy as np
import torch.nn.functional as F
from tqdm.notebook import tqdm
from transformers import CLIPProcessor, CLIPModel, Blip2Processor, Blip2ForConditionalGeneration
from IPython.display import display, HTML

# --- 1. CONFIGURATION & DEVICES ---
print("Checking GPU availability...")
if torch.cuda.device_count() >= 2:
    DEVICE_YOLO = "cuda:0"
    DEVICE_BLIP = "cuda:1"
    BLIP_DEVICE_MAP = {"": 1}
else:
    DEVICE_YOLO = "cuda:0" if torch.cuda.is_available() else "cpu"
    DEVICE_BLIP = DEVICE_YOLO
    BLIP_DEVICE_MAP = {"": 0}

BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
META_PATH = "/kaggle/input/datasets/dveers/vr-final-ds/gallery_metadata.json"
SEEDS = [543, 45, 56]
ALPHAS = [0.5, 0.8]
k_values = [5, 10, 15]
metrics_names = ['Recall', 'NDCG', 'mAP']

# --- 2. LOAD METADATA & METRICS ---
with open(META_PATH, "r", encoding='utf-8') as f:
    metadata = json.load(f)

def calculate_metrics(retrieved_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    results = {'Recall': {}, 'NDCG': {}, 'mAP': {}}
    for k in k_values:
        top_k = retrieved_item_ids[:k]
        relevance = [1 if item == ground_truth_id else 0 for item in top_k]
        results['Recall'][k] = 1 if sum(relevance) > 0 else 0
        dcg = sum([rel / np.log2(idx + 2) for idx, rel in enumerate(relevance)])
        results['NDCG'][k] = dcg / 1.0  
        precisions = [hits / (i + 1) for i, (rel, hits) in enumerate(zip(relevance, np.cumsum(relevance))) if rel == 1]
        results['mAP'][k] = sum(precisions) / min(len(top_k), sum(relevance) + 1e-6) if precisions else 0
    return results

if 'blip_model' in globals() and 'blip_processor' in globals():
    print(f"BLIP-2 is already loaded in memory! Skipping download/initialization.")
else:
    print(f"Loading Frozen BLIP-2 to {DEVICE_BLIP}...")
    blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
    blip_model = Blip2ForConditionalGeneration.from_pretrained(
        "Salesforce/blip2-opt-2.7b", 
        torch_dtype=torch.float16, 
        device_map=BLIP_DEVICE_MAP
    )

# --- 4. OPTIMIZED EVALUATION LOOP ---
print("\nRunning ULTRA-FAST Evaluation for Part C (Processing both Alphas simultaneously)...")

# Trackers for both alphas
seed_results = {alpha: {m: {k: [] for k in k_values} for m in metrics_names} for alpha in ALPHAS}

for seed in SEEDS:
    print(f"\n" + "="*50 + f"\nEvaluating Seed {seed}\n" + "="*50)
    model_path = f"/kaggle/input/models/dveers/finetuned-clip-vr-final/pytorch/default/2/finetuned_clip_full_{seed}"
    
    # Load the specific Fine-Tuned CLIP for this seed
    ft_clip_processor = CLIPProcessor.from_pretrained(model_path)
    ft_clip_model = CLIPModel.from_pretrained(model_path).to(DEVICE_YOLO)
    
    # Load BOTH FAISS indices for this seed into memory
    index_05 = faiss.read_index(f"/kaggle/input/datasets/dveers/vr-final-ds/all_faiss_indices/gallery_index_finetuned_{seed}_alpha05.faiss")
    index_08 = faiss.read_index(f"/kaggle/input/datasets/dveers/vr-final-ds/all_faiss_indices/gallery_index_finetuned_{seed}_alpha08.faiss")
    
    valid_queries = 0
    total_metrics = {alpha: {m: {k: 0 for k in k_values} for m in metrics_names} for alpha in ALPHAS}
    
    for idx, row in tqdm(query_df.iterrows(), total=len(query_df)):
        abs_path = os.path.join(BASE_DIR, "img", row['image_name'])
        if not os.path.exists(abs_path): continue
        valid_queries += 1
        
        cropped_query = get_cropped_image(abs_path)
        with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            
            # 1. FAISS Query Embed (GPU 0) - Calculate ONCE
            clip_inputs = ft_clip_processor(images=cropped_query, return_tensors="pt").to(DEVICE_YOLO)
            vision_out = ft_clip_model.vision_model(pixel_values=clip_inputs.pixel_values)
            query_embed = ft_clip_model.visual_projection(vision_out.pooler_output)
            query_embed = query_embed / query_embed.norm(p=2, dim=-1, keepdim=True)
            query_np = query_embed.to(torch.float32).cpu().numpy()
            
            # 2. Search BOTH indices simultaneously
            _, idx_05 = index_05.search(query_np, 15)
            _, idx_08 = index_08.search(query_np, 15)
            
            candidates_05 = [metadata[i] for i in idx_05[0]]
            candidates_08 = [metadata[i] for i in idx_08[0]]
            
            # 3. Combine unique candidates to save BLIP-2 compute time!
            unique_candidates_dict = {c['faiss_id']: c for c in candidates_05 + candidates_08}
            unique_candidates = list(unique_candidates_dict.values())
            
            # 4. BLIP-2 Batched Re-Ranking (GPU 1) - Calculate ONCE on the unique set
            pixel_values = blip_processor(images=cropped_query, return_tensors="pt").pixel_values.to(DEVICE_BLIP, torch.float16)
            pixel_values = pixel_values.expand(len(unique_candidates), -1, -1, -1)
            
            captions = [c['generated_caption'] for c in unique_candidates]
            text_inputs = blip_processor.tokenizer(captions, return_tensors="pt", padding=True, truncation=True).to(DEVICE_BLIP)
            
            input_ids, attention_mask = text_inputs.input_ids, text_inputs.attention_mask
            labels = input_ids.clone()
            labels[labels == blip_processor.tokenizer.pad_token_id] = -100
            
            outputs = blip_model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            
            logits = outputs.logits
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_matrix = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction='none')
            loss_matrix = loss_matrix.view(shift_labels.size())
            mask = (shift_labels != -100)
            losses = ((loss_matrix * mask).sum(dim=1) / mask.sum(dim=1)).tolist()
            
            # Create a fast lookup dictionary for the scores
            score_lookup = {unique_candidates[i]['faiss_id']: losses[i] for i in range(len(unique_candidates))}
            
            # 5. Split, sort, and calculate metrics for Alpha 0.5
            reranked_05 = [(score_lookup[c['faiss_id']], c) for c in candidates_05]
            reranked_05.sort(key=lambda x: x[0])
            final_05 = [item[1]['item_id'] for item in reranked_05]
            metrics_05 = calculate_metrics(final_05, row['item_id'], k_values)
            for m in metrics_names:
                for k in k_values: total_metrics[0.5][m][k] += metrics_05[m][k]
                    
            # 6. Split, sort, and calculate metrics for Alpha 0.8
            reranked_08 = [(score_lookup[c['faiss_id']], c) for c in candidates_08]
            reranked_08.sort(key=lambda x: x[0])
            final_08 = [item[1]['item_id'] for item in reranked_08]
            metrics_08 = calculate_metrics(final_08, row['item_id'], k_values)
            for m in metrics_names:
                for k in k_values: total_metrics[0.8][m][k] += metrics_08[m][k]
            
    # Tally the seed averages for both alphas
    for alpha in ALPHAS:
        for m in metrics_names:
            for k in k_values:
                seed_results[alpha][m][k].append(total_metrics[alpha][m][k] / valid_queries)
            
    # CLEAR THIS SPECIFIC SEED'S MODELS BEFORE LOADING THE NEXT ONE
    del ft_clip_model, ft_clip_processor, index_05, index_08
    gc.collect()
    if torch.cuda.device_count() >= 2:
        with torch.cuda.device(0): torch.cuda.empty_cache() 
    else:
        torch.cuda.empty_cache()

# --- 5. OUTPUT FINAL RESULTS FOR BOTH ALPHAS ---
for alpha in ALPHAS:
    print(f"\nFINAL RESULTS: Fine-Tuned CLIP + BLIP-2 (Alpha = {alpha})")
    final_table = {}
    for m in metrics_names:
        final_table[m] = {}
        for k in k_values:
            mean_val = np.mean(seed_results[alpha][m][k])
            std_val = np.std(seed_results[alpha][m][k])
            final_table[m][f"@{k}"] = f"{mean_val:.4f} ± {std_val:.4f}"

    results_df = pd.DataFrame(final_table)
    display(HTML(results_df.to_html(classes='table table-striped table-bordered text-center', float_format="%.4f")))

Checking GPU availability...
Loading Frozen BLIP-2 to cuda:1...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

...

  0%|          | 0/14218 [00:00<?, ?it/s]


FINAL RESULTS: Fine-Tuned CLIP + BLIP-2 (Alpha = 0.5)


,Recall,NDCG,mAP
@5,0.3148 ± 0.0166,0.2331 ± 0.0151,0.1556 ± 0.0091
@10,0.5228 ± 0.0235,0.3589 ± 0.0226,0.1757 ± 0.0095
@15,0.6668 ± 0.0238,0.4610 ± 0.0268,0.1794 ± 0.0092



FINAL RESULTS: Fine-Tuned CLIP + BLIP-2 (Alpha = 0.8)


,Recall,NDCG,mAP
@5,0.3562 ± 0.0136,0.2667 ± 0.0131,0.1744 ± 0.0079
@10,0.5855 ± 0.0161,0.4149 ± 0.0192,0.1960 ± 0.0069
@15,0.7400 ± 0.0134,0.5345 ± 0.0216,0.1995 ± 0.0066
